# PTM-SAE activation diagnosis (CPU only)

Why is explained variance low at small k? Samples cached ESM-2 650M layer-24 activations (a handful of shards per partition, hydrated from the public Hub dataset) and reports their geometry with `scripts/diagnose_activations.py`: how concentrated the variance is, the PCA spectrum, how much the amino acid alone explains, chain-end effects, and the explained variance an ideal k-sparse code reaches. Nothing is trained; no GPU is needed (push without `--gpu`).

Parameters (environment variables, set by `scripts/kaggle_run.py push --param`): `PTM_SAE_DIAG_TOKENS` (tokens from `discovery_train`; a third of that from `discovery_val`) and `PTM_SAE_DIAG_SHARDS` (shards sampled per partition).

Results land in `diagnosis/*.json` and `logs/`.

In [ ]:
# 1. Environment & GPU verification
# Deliberately imports no numpy/torch: dependencies are installed in step 3, and a package imported
# before pip upgrades it stays at the old version inside this kernel (later imports then fail).
import platform
import subprocess

print("python", platform.python_version())
gpu = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else "no NVIDIA GPU visible (CPU run)")

In [ ]:
# 2. Repository Setup (same on a local PC, a persistent remote box, Kaggle and Colab)
# Finds the checkout by walking up from the kernel's working directory, so it also works when this
# notebook is opened from notebooks/. Only a runtime with no checkout (fresh Kaggle/Colab) clones;
# an existing checkout is fast-forwarded (local edits are auto-stashed, never discarded) on its
# current branch -- switch branches yourself with `git checkout` in a terminal.
import os
import subprocess
import sys
from pathlib import Path

BRANCH = os.environ.get("PTM_SAE_BRANCH", "main")  # used only when cloning a fresh runtime
PULL = os.environ.get("PTM_SAE_PULL", "1") == "1"  # fast-forward an existing checkout (PTM_SAE_PULL=0 to skip)
REPO = "github.com/Mumu-kun/PTM-SAE.git"

here = Path.cwd().resolve()
root = next(
    (p for p in (here, *here.parents) if (p / "pyproject.toml").is_file() and (p / "src/ptm_sae").is_dir()),
    None,
)

if root is None:
    token = os.environ.get("GH_TOKEN")  # private repo: also read Kaggle/Colab secrets
    try:
        from kaggle_secrets import UserSecretsClient

        token = token or UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    try:  # an API-pushed kernel has no attached Secrets: read the private secrets dataset instead
        import json

        found = [p for pattern in ("*/ptm_sae_secrets.json", "*/*/ptm_sae_secrets.json") for p in Path("/kaggle/input").glob(pattern)]
        token = token or json.loads(found[0].read_text(encoding="utf-8")).get("GH_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata

        token = token or userdata.get("GH_TOKEN")
    except Exception:
        pass

    auth = f"x-access-token:{token}@" if token else ""
    root = (Path("/kaggle/working") if Path("/kaggle").exists() else here) / "PTM-SAE"
    print(f"Cloning {BRANCH} into {root}...")
    try:
        subprocess.run(["git", "clone", "--branch", BRANCH, f"https://{auth}{REPO}", str(root)], check=True, capture_output=True)
    except subprocess.CalledProcessError:
        raise RuntimeError(f"git clone of branch {BRANCH!r} failed: check the branch name and that the GH_TOKEN secret is attached to this notebook") from None
elif PULL:
    print(f"Fast-forwarding existing checkout at {root}...")
    subprocess.run(["git", "pull", "--ff-only", "--autostash"], cwd=root, check=False)

os.chdir(root)
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from ptm_sae import runtime

print(f"Platform: {runtime.detect_platform()} | repo: {root} | data root: {runtime.resolve_data_root()}")

In [ ]:
# 3. Install Dependencies (single-sourced from pyproject.toml)
# Local/remote boxes manage their environment with `uv sync --extra notebook` in a terminal (locked
# versions, CUDA torch build kept intact), so there this cell only checks it. Kaggle/Colab ship a
# CUDA-matched torch/transformers, so there everything else is pip-installed (retried: a flaky mirror
# must not end a long run) and those two are left alone. A failure is recorded below, not raised.
import importlib.metadata
import subprocess
import sys
import time

EXTRAS = ()
INSTALL_OK = True
# Packages this kernel already imported must not be changed by pip underneath it (the running process
# would be half-old, half-new and later imports fail): remember their versions to check afterwards.
IMPORTED_BEFORE = {n: importlib.metadata.version(n) for n in ("numpy", "pandas", "pyarrow", "scipy", "torch") if n in sys.modules}

if runtime.detect_platform() == "local":
    missing = runtime.missing_requirements(extras=EXTRAS)
    INSTALL_OK = not missing
    if missing:
        print(f"Missing {missing}: run `uv sync --extra notebook` in a terminal, then restart the kernel.")
else:
    requirements = runtime.cloud_requirements(extras=EXTRAS)
    for attempt in range(1, 4):
        deps = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements], capture_output=True, text=True)
        local = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps", "--ignore-requires-python"], capture_output=True, text=True)
        if deps.returncode == 0 and local.returncode == 0:
            break
        print(f"pip attempt {attempt} failed: {(deps.stderr + local.stderr)[-400:]}")
        time.sleep(10 * attempt)
    else:
        INSTALL_OK = False
changed = [n for n, version in IMPORTED_BEFORE.items() if importlib.metadata.version(n) != version]
if changed:
    INSTALL_OK = False
    print(f"RESTART THE KERNEL: {changed} were imported before pip changed them; later imports would fail.")
print("dependencies ready:", INSTALL_OK)

In [ ]:
# 4. Parameters, run state and preflight
import os
import sys
from pathlib import Path

MAX_TOKENS = int(os.environ.get("PTM_SAE_DIAG_TOKENS", "600000"))  # tokens sampled from discovery_train; a third of that from discovery_val
N_SHARDS = int(os.environ.get("PTM_SAE_DIAG_SHARDS", "12"))  # shards sampled per partition: more shards, a more representative sample

DATA_ROOT = runtime.resolve_data_root()
LOG_DIR = DATA_ROOT / "logs"

state = runtime.RunState()
with state.cell("install"):
    if not INSTALL_OK:
        raise RuntimeError("dependency install failed (see the install cell output)")

with state.cell("preflight"):
    state.require("install")
    report = runtime.preflight(need_gpu=False, need_hub_write=False, min_free_gb=10)
    if not report.ok:
        raise RuntimeError(f"preflight failed: {report.failures()}")
print(f"tokens={MAX_TOKENS:,} shards={N_SHARDS} data root={DATA_ROOT}")

In [ ]:
# 5. Diagnosis of both partitions: a subprocess with a log, so a crash cannot take the notebook down
with state.cell("diagnose"):
    state.require("preflight")
    for partition, tokens in (("discovery_train", MAX_TOKENS), ("discovery_val", MAX_TOKENS // 3)):
        log = LOG_DIR / f"diagnose_{partition}.log"
        cmd = [
            sys.executable, "-u", "scripts/diagnose_activations.py", "--partition", partition,
            "--max-tokens", str(tokens), "--shards", str(N_SHARDS), "--out", str(DATA_ROOT / "diagnosis" / f"{partition}.json"),
        ]  # fmt: skip
        code = runtime.run_logged(cmd, log, timeout_s=state.time_left())
        if code != 0:
            raise RuntimeError(f"diagnosis of {partition} exited with {code}; see {log}")

In [ ]:
# 6. Run summary -- always runs: what completed, what did not, and the files that can be salvaged
try:
    state.print_summary(artifact_dirs=(DATA_ROOT / "diagnosis", LOG_DIR))
except Exception as exc:  # noqa: BLE001 -- the summary must never be what fails the run
    print(f"summary failed: {type(exc).__name__}: {exc}")